In [1]:
import os
import sys
from datetime import datetime
import itertools

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns_infinite
import infinite
reload(plotting)
reload(pinns_infinite)
reload(infinite)
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns_infinite import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns_infinite import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(42)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')


class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
 
# Create one timestamp for the entire experiment batch
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

results_dir = f"results_accuracy_efficiency_{timestamp}"

print(f"Results will be saved to: {results_dir}")


# Optimal hyperparameters found in 02_hyperparameter_tunning (results_mlp_infinite_optuna_2026-09-20_08-52-35)
optimal_activation = Sine()
optimal_adam_lr = 1e-3

# Configurations to sweep: L (hidden layers) x width (hidden units), keeping the rest of the hyperparameters optimal
L_values = [1, 2, 3]
width_values = [15, 90, 104]
mlp_configurations = [
    {"L": L, "width": width}
    for L in L_values
    for width in width_values
]


# Loop through each matched configuration
for cfg in mlp_configurations:

    layers = cfg["L"]
    width = cfg["width"]

    print(
        f"\n--- Running MLP Experiment: "
        f"Layers (L)={layers}, Width (N)={width} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="MLP",
            hidden_layers=layers,
            hidden_units=width,
            activation=optimal_activation,
            adam_lr=optimal_adam_lr,
            device=device,
            adam_iters=2000,
            lbfgs_iters=2000,
            results_dir=results_dir,
        )

        print(
            f"Success! Time: {compute_time:.2f}s | "
            f"Err U: {err_u:.3e} | Err K: {err_k:.3e}"
        )

    except Exception as e:
        print(
            f"Experiment failed for "
            f"Layers={layers}, Width={width} with error: {e}"
        )

Results will be saved to: results_accuracy_efficiency_2026-09-20_17-10-10

--- Running MLP Experiment: Layers (L)=1, Width (N)=15 ---


/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



[MLP] L=1, N=15 | Params: 602 | Mean Err: 8.804e-01 | Saved to 'results_accuracy_efficiency_2026-09-20_17-10-10/'.
Success! Time: 74.32s | Err U: 2.763e-01 | Err K: 1.484e+00

--- Running MLP Experiment: Layers (L)=1, Width (N)=90 ---

[MLP] L=1, N=90 | Params: 17,102 | Mean Err: 4.748e-02 | Saved to 'results_accuracy_efficiency_2026-09-20_17-10-10/'.
Success! Time: 13.56s | Err U: 8.894e-02 | Err K: 6.010e-03

--- Running MLP Experiment: Layers (L)=1, Width (N)=104 ---

[MLP] L=1, N=104 | Params: 22,674 | Mean Err: 4.640e-02 | Saved to 'results_accuracy_efficiency_2026-09-20_17-10-10/'.
Success! Time: 14.57s | Err U: 8.642e-02 | Err K: 6.376e-03

--- Running MLP Experiment: Layers (L)=2, Width (N)=15 ---

[MLP] L=2, N=15 | Params: 1,082 | Mean Err: 7.174e-02 | Saved to 'results_accuracy_efficiency_2026-09-20_17-10-10/'.
Success! Time: 85.97s | Err U: 2.129e-02 | Err K: 1.222e-01

--- Running MLP Experiment: Layers (L)=2, Width (N)=90 ---

[MLP] L=2, N=90 | Params: 33,482 | Mean Err: 